# Day 26 · 多模态偏好数据构造

**配套讲义**: [`days/day-26.md`](../days/day-26.md) ｜ **本地可跑，不需要 GPU**

从 W4 的 bad case 反向构造 (chosen, rejected) 对，凑够 3k 对；并亲手构造**同图不同答**和**同答不同图**两种多模态特有的偏好对。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w5.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys
print("python:", sys.version.split()[0])
for m in ("numpy", "PIL", "yaml", "pandas"):
    try:
        mod = __import__(m)
        print(f"  {m:7s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"  {m:7s} ❌ 缺 → pip install {m}")
print("\n→ 本机没 GPU 不影响今天：今天只用纯 Python / numpy")

## 1. 从 bad case 造偏好对

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.train.dpo_loss",
                    "--from-badcases", "reports/bad_cases.jsonl"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout or r.stderr)

## 2. 造对比式偏好对（多模态特有两种）

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.train.dpo_loss",
                    "--contrastive", "data/processed/clean.jsonl", "--n-pairs", "1000"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout[-1500:] or r.stderr[-1500:])

## 3. 抽 5 对肉眼检查「差异是可学习的吗」

**这是今天最关键的判断**。差异太小 → 学不到；差异太大（一个全错一个全对）→ 也没用，
模型会去学那些显而易见的差异而不是你想要的细节。

In [ ]:
import json, random
from pathlib import Path

for name in ("dpo_train", "dpo_contrastive"):
    p = Path(f"../data/processed/{name}.jsonl")
    if not p.exists():
        print(f"{name}: 还没生成"); continue
    rows = [json.loads(l) for l in p.read_text().splitlines() if l.strip()]
    print("=" * 72)
    print(f"{name}  共 {len(rows)} 对")
    for r in random.sample(rows, min(2, len(rows))):
        print("-" * 72)
        print("图  :", r.get("images"))
        print("问  :", str(r.get("prompt"))[:80])
        print("✓chosen  :", str(r.get("chosen"))[:110])
        print("✗rejected:", str(r.get("rejected"))[:110])
        print("来源:", r.get("source"), "| 类型:", r.get("error_type", r.get("pair_type", "-")))

## 4. 落笔：三行判断

- 哪些对「差异太小，学不到」？
- 哪些对「差异太大，学到的是废话」？
- 你想补哪一类？

In [ ]:
judgement = """
差异太小的例子：
差异太大的例子：
我打算补的数据：
"""
print(judgement)

## 验收清单

- [ ] `dpo_train.jsonl` 有实打实的偏好对（不是 0 对）
- [ ] 抽检 30 对，逐对判断「rejected 确实更差，且差异是**可学习的**」
- [ ] 三种类型的偏好对都有：同图不同答 / 同答不同图 / 格式或工具调用
- [ ] 能说清哪一类偏好对治的是「幻觉」这个具体病

**卡住了？** 回看 [`days/day-26.md`](../days/day-26.md) 第五节「容易踩的坑」。

> **明天**：`days/day-27.md` —— 真正跑 DPO（回云上）